### `Ideia do notebook:`

Usar roBERTa para prever peso, altura, largura e comprimento a partir do título.

### Setup inicial

In [11]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from sklearn.preprocessing import StandardScaler

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o device: {device}")

Usando o device: cpu


Baixando modelo e executando exemplo fornecido no hugging face. O tensor unidimensional na saída (produto interno) representa a "afinidade" da query com os possibilidades.

In [12]:
tokenizer = AutoTokenizer.from_pretrained("hyp1231/blair-roberta-base")
model = AutoModel.from_pretrained("hyp1231/blair-roberta-base")

language_context = 'I need a product that can scoop, measure, and rinse grains without the need for multiple utensils and dishes. It would be great if the product has measurements inside and the ability to rinse and drain all in one. I just have to be careful not to pour too much accidentally.'
item_metadata = [
  'Talisman Designs 2-in-1 Measure Rinse & Strain | Holds up to 2 Cups | Food Strainer | Fruit Washing Basket | Strainer & Colander for Kitchen Sink | Dishwasher Safe - Dark Blue. The Measure Rinse & Strain by Talisman Designs is a 2-in-1 kitchen colander and strainer that will measure and rinse up to two cups. Great for any type of food from rice, grains, beans, fruit, vegetables, pasta and more. After measuring, fill with water and swirl to clean. Strain then pour into your pot, pan, or dish. The convenient size is easy to hold with one hand and is compact to fit into a kitchen cabinet or pantry. Dishwasher safe and food safe.',
  'FREETOO Airsoft Gloves Men Tactical Gloves for Hiking Cycling Climbing Outdoor Camping Sports (Not Support Screen Touch).'
]
texts = [language_context] + item_metadata

inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

# Get the embeddings
with torch.no_grad():
    embeddings = model(**inputs, return_dict=True).last_hidden_state[:, 0]
    embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)

print(embeddings[0] @ embeddings[1])    # tensor(0.8564)
print(embeddings[0] @ embeddings[2])    # tensor(0.5741)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tensor(0.8564)
tensor(0.5741)


### Setup Transferlearning

criando datasets pandas

In [14]:
caminho_csv = "../data/cubagem_40k_amazon.csv" 
df = pd.read_csv(caminho_csv)

CATEGORIA_ALVO = "Beauty & Personal Care" 

df = df[df['categories'].str.contains(CATEGORIA_ALVO, na=False, case=False)]

df_train = df[df['split'] == 'train']
df_val = df[df['split'] == 'val']

colunas_alvo = ['length_cm', 'width_cm', 'height_cm', 'weight_g']
df_train = df_train.dropna(subset=['title'] + colunas_alvo)
df_val = df_val.dropna(subset=['title'] + colunas_alvo)

print(f"Total de produtos de treino: {len(df_train)}")
print(f"Total de produtos de validação: {len(df_val)}")

Total de produtos de treino: 3354
Total de produtos de validação: 839


Fazendo normalização (scaler) e carregando dataloaders.

In [15]:
# ---------------------------------------------------------
# SCALER
# ---------------------------------------------------------
textos_train = df_train['title'].values
targets_train = df_train[colunas_alvo].values

textos_val = df_val['title'].values
targets_val = df_val[colunas_alvo].values

scaler = StandardScaler()

targets_train_escalonados = scaler.fit_transform(targets_train)
targets_val_escalonados = scaler.transform(targets_val) # usamos média e var de treino

# ---------------------------------------------------------
# CRIAÇÃO DOS DATALOADERS
# ---------------------------------------------------------
class CubagemDataset(Dataset):
    def __init__(self, textos, targets, tokenizer, max_len=128):
        self.textos = textos
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        texto = str(self.textos[idx])
        target = self.targets[idx]

        encoding = self.tokenizer(
            texto,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target, dtype=torch.float32)
        }

# Dataset de Treino
dataset_train = CubagemDataset(textos_train, targets_train_escalonados, tokenizer)
dataloader = DataLoader(dataset_train, batch_size=16, shuffle=True)

# Dataset de Validação
dataset_val = CubagemDataset(textos_val, targets_val_escalonados, tokenizer)
val_dataloader = DataLoader(dataset_val, batch_size=16, shuffle=False)

In [16]:
# ---------------------------------------------------------
# ARQUITETURA DO MODELO
# ---------------------------------------------------------
class ModeloCubagem(nn.Module):
    def __init__(self, model_name):
        super(ModeloCubagem, self).__init__()
        self.base_model = AutoModel.from_pretrained(model_name)
        
        # CONGELAMENTO DOS PESOS (Fase de Aquecimento)
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        hidden_size = self.base_model.config.hidden_size
        
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 4) # 4 outputs: length_cm, width_cm, height_cm, weight_g
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.mlp(cls_embedding)

modelo = ModeloCubagem("hyp1231/blair-roberta-base").to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
# ---------------------------------------------------------
# TREINAMENTO
# ---------------------------------------------------------

criterion = nn.HuberLoss() # é um MSE com L1 nos outliers
optimizer = optim.Adam(modelo.mlp.parameters(), lr=1e-3)

epochs = 5

for epoch in range(epochs):
    
    # ==========================================
    # TREINAMENTO
    # ==========================================
    modelo.train()
    modelo.base_model.eval() # Mantém o base model estático
    
    train_loss = 0
    
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets_batch = batch['targets'].to(device)
        
        optimizer.zero_grad()
        
        predicoes = modelo(input_ids, attention_mask)
        loss = criterion(predicoes, targets_batch)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(dataloader)

    # ==========================================
    # VALIDAÇÃO E EXEMPLOS
    # ==========================================
    modelo.eval() # Coloca a MLP em modo de avaliação também
    val_loss = 0
    exemplos_capturados = False # Flag para pegar apenas 1 batch para visualização
    
    # torch.no_grad() desliga o cálculo de gradientes, poupando memória e tempo
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets_batch = batch['targets'].to(device)
            
            predicoes = modelo(input_ids, attention_mask)
            loss = criterion(predicoes, targets_batch)
            val_loss += loss.item()
            
            # Captura o primeiro batch de validação para mostrar exemplos
            if not exemplos_capturados:
                # Traz os tensores para a CPU e converte para NumPy
                preds_np = predicoes.cpu().numpy()
                targets_np = targets_batch.cpu().numpy()
                textos_exemplo = batch.get('texto_original', ['Produto Exemplo'] * len(preds_np)) 
                
                # Reverte a escala para os valores originais em cm e gramas!
                preds_reais = scaler.inverse_transform(preds_np)
                targets_reais = scaler.inverse_transform(targets_np)
                
                exemplos_capturados = True
                
    avg_val_loss = val_loss / len(val_dataloader)
    
    # ==========================================
    # EXEMPLOS DE VALIDAÇÃO A CADA ÉPOCA
    # ==========================================
    print(f"\n[Época {epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    print("-" * 60)
    print("Visualizando predições (Validação):")

    for i in range(min(2, len(preds_reais))):
        print(f"  > Exemplo {i+1}:")
        print(f"    Real -> Comp: {targets_reais[i][0]:.1f}cm | Larg: {targets_reais[i][1]:.1f}cm | Alt: {targets_reais[i][2]:.1f}cm | Peso: {targets_reais[i][3]:.1f}g")
        print(f"    Pred -> Comp: {preds_reais[i][0]:.1f}cm | Larg: {preds_reais[i][1]:.1f}cm | Alt: {preds_reais[i][2]:.1f}cm | Peso: {preds_reais[i][3]:.1f}g")
    print("=" * 60)


[Época 1/5] Train Loss: 0.1830 | Val Loss: 0.1725
------------------------------------------------------------
Visualizando predições (Validação):
  > Exemplo 1:
    Real -> Comp: 7.0cm | Larg: 7.0cm | Alt: 1.3cm | Peso: 64.1g
    Pred -> Comp: 19.5cm | Larg: 12.2cm | Alt: 4.1cm | Peso: 176.2g
  > Exemplo 2:
    Real -> Comp: 24.1cm | Larg: 14.0cm | Alt: 2.5cm | Peso: 204.1g
    Pred -> Comp: 21.3cm | Larg: 15.0cm | Alt: 7.3cm | Peso: 769.5g

[Época 2/5] Train Loss: 0.1620 | Val Loss: 0.1632
------------------------------------------------------------
Visualizando predições (Validação):
  > Exemplo 1:
    Real -> Comp: 7.0cm | Larg: 7.0cm | Alt: 1.3cm | Peso: 64.1g
    Pred -> Comp: 15.3cm | Larg: 10.4cm | Alt: 3.8cm | Peso: 85.7g
  > Exemplo 2:
    Real -> Comp: 24.1cm | Larg: 14.0cm | Alt: 2.5cm | Peso: 204.1g
    Pred -> Comp: 20.0cm | Larg: 13.1cm | Alt: 7.1cm | Peso: 441.7g

[Época 3/5] Train Loss: 0.1556 | Val Loss: 0.1611
--------------------------------------------------------